# Qwen3.5-4B: una consulta y sus métricas

Usa el checkpoint ya descargado en `../Lora Scripts/models/` y el entorno local `.venv`.
El modelo se carga en GPU al ejecutar: los pesos descargados permanecen en disco.
Abrir con ese kernel **dentro de una asignación Slurm GPU**, siguiendo `AGENTS.md`.
No hay entrenamiento ni adaptadores. Ejecutar de arriba hacia abajo.

Para instalar las dependencias: ejecutar `uv sync --locked` desde esta carpeta
y seleccionar el intérprete de `qwen_test/.venv/bin/python` en Jupyter.

## 1. Preparar el modelo local

In [1]:
import json
import os
import statistics
import subprocess
import threading
import time
from pathlib import Path

import torch
from transformers import AutoTokenizer, BitsAndBytesConfig, Qwen3_5ForConditionalGeneration

if not os.environ.get("SLURM_JOB_ID") or not torch.cuda.is_available():
    raise RuntimeError("En Khipu ejecuta este notebook dentro de un job Slurm con GPU.")

model_roots = [Path.cwd(), *Path.cwd().parents, Path.cwd().parent / "Lora Scripts"]
project_dir = next((p for p in model_roots
                    if (p / "models/model_manifest.json").is_file()), None)
if project_dir is None:
    raise FileNotFoundError("No se encontró models/model_manifest.json del proyecto.")
manifest = json.loads((project_dir / "models/model_manifest.json").read_text())
assert manifest["model_id"] == "Qwen/Qwen3.5-4B"
model_path = Path(manifest["snapshot_path"])
if not model_path.is_dir():
    raise FileNotFoundError(f"Checkpoint local no encontrado: {model_path}")

precision = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
load_start = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model = Qwen3_5ForConditionalGeneration.from_pretrained(
    model_path,
    local_files_only=True,
    device_map={"": 0},
    dtype=precision,
    attn_implementation="sdpa",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=precision,
    ),
).eval()
model.requires_grad_(False)
torch.cuda.synchronize()
load_seconds = time.perf_counter() - load_start
print(f"Modelo: {manifest['model_id']} | GPU: {torch.cuda.get_device_name(0)}")
print(f"Carga del tokenizer y modelo: {load_seconds:.2f} s")

/home/jeffrey.monja/Github/2026-II/PlanIA/S10/Lora Scripts/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/723 [00:00<02:18,  5.23it/s]

Loading weights:   2%|▏         | 13/723 [00:00<00:13, 51.93it/s]

Loading weights:   5%|▍         | 36/723 [00:00<00:05, 116.59it/s]

...

Loading weights:  85%|████████▌ | 617/723 [00:02<00:00, 385.02it/s]

Loading weights:  91%|█████████▏| 660/723 [00:03<00:00, 397.64it/s]

Loading weights:  99%|█████████▉| 717/723 [00:03<00:00, 447.97it/s]

Loading weights: 100%|██████████| 723/723 [00:03<00:00, 229.79it/s]

Modelo: Qwen/Qwen3.5-4B | GPU: NVIDIA RTX A6000
Carga del tokenizer y modelo: 7.39 s


## 2. Hacer un prompt sencillo

Cambia `prompt` y ejecuta esta celda y la de resultados para otra consulta. La generación es
sin muestreo y con thinking desactivado. El tiempo de inferencia incluye preparar
la entrada, generar y decodificar; la carga del modelo se mide por separado.

El porcentaje de uso se muestrea con `nvidia-smi` cada 200 ms. En una GPU compartida
incluye la actividad de otros procesos; no equivale al uso exclusivo del notebook.

In [2]:
prompt = "Explica en dos frases qué es un modelo de lenguaje."
max_new_tokens = 128

gpu_samples = []
stop_sampling = threading.Event()
gpu_id = os.environ.get("CUDA_VISIBLE_DEVICES", "0").split(",")[0]
sampling_errors = []

def sample_gpu():
    while not stop_sampling.is_set():
        try:
            result = subprocess.run(
                ["nvidia-smi", f"--id={gpu_id}", "--query-gpu=utilization.gpu",
                 "--format=csv,noheader,nounits"],
                capture_output=True, text=True, check=True, timeout=2,
            )
            gpu_samples.append(float(result.stdout.strip()))
        except (OSError, ValueError, subprocess.SubprocessError) as exc:
            sampling_errors.append(getattr(exc, "stderr", None) or str(exc))
        stop_sampling.wait(0.2)

sampler = threading.Thread(target=sample_gpu, daemon=True)
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
free_before, total_gpu_memory = torch.cuda.mem_get_info()
inference_start = time.perf_counter()
sampler.start()
try:
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=True, add_generation_prompt=True, enable_thinking=False,
        return_dict=True, return_tensors="pt",
    ).to("cuda:0")
    torch.cuda.synchronize()
    generation_start = time.perf_counter()
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=sorted({tokenizer.eos_token_id,
                                tokenizer.convert_tokens_to_ids("<|endoftext|>")}),
        )
    torch.cuda.synchronize()
    generation_seconds = time.perf_counter() - generation_start
    input_tokens = inputs.input_ids.shape[1]
    generated_ids = outputs[0, input_tokens:]
    output_tokens = len(generated_ids)
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    inference_seconds = time.perf_counter() - inference_start
finally:
    stop_sampling.set()
    sampler.join(timeout=3)

free_after, _ = torch.cuda.mem_get_info()
metrics = {
    "model": manifest["model_id"],
    "model_revision": manifest["revision"],
    "gpu": torch.cuda.get_device_name(0),
    "input_tokens": input_tokens,
    "output_tokens": output_tokens,
    "total_tokens": input_tokens + output_tokens,
    "model_load_seconds": round(load_seconds, 3),
    "generation_seconds": round(generation_seconds, 3),
    "inference_seconds": round(inference_seconds, 3),
    "output_tokens_per_second": round(output_tokens / generation_seconds, 2),
    "cuda_allocated_gib": round(torch.cuda.memory_allocated() / 2**30, 3),
    "cuda_peak_allocated_gib": round(torch.cuda.max_memory_allocated() / 2**30, 3),
    "cuda_peak_reserved_gib": round(torch.cuda.max_memory_reserved() / 2**30, 3),
    "device_total_memory_gib": round(total_gpu_memory / 2**30, 3),
    "device_free_before_gib": round(free_before / 2**30, 3),
    "device_free_after_gib": round(free_after / 2**30, 3),
    "device_gpu_utilization_mean_percent": round(statistics.mean(gpu_samples), 1) if gpu_samples else None,
    "device_gpu_utilization_max_percent": max(gpu_samples) if gpu_samples else None,
    "gpu_utilization_samples": len(gpu_samples),
    "gpu_utilization_error": sampling_errors[-1].strip() if not gpu_samples and sampling_errors else None,
}

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


## 3. Ver la respuesta y las métricas

Tokens totales = entrada (incluido el chat template) + salida (incluido el token
final si se generó). La primera consulta puede incluir inicialización de kernels.
VRAM asignada/pico corresponde a este proceso; VRAM libre y utilización corresponden
al dispositivo compartido. `null` significa que `nvidia-smi` no proporcionó el dato.

In [3]:
print("PROMPT:", prompt)
print("\nRESPUESTA:\n", response)
print("\nMÉTRICAS:\n", json.dumps(metrics, ensure_ascii=False, indent=2))

PROMPT: Explica en dos frases qué es un modelo de lenguaje.

RESPUESTA:
 Un modelo de lenguaje es un sistema computacional entrenado con grandes volúmenes de texto que aprende las patrones y relaciones del idioma humano. Su objetivo principal es predecir y generar la siguiente palabra o frase en una secuencia dada, permitiendo realizar tareas como responder preguntas, redactar textos o traducir idiomas.

MÉTRICAS:
 {
  "model": "Qwen/Qwen3.5-4B",
  "model_revision": "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a",
  "gpu": "NVIDIA RTX A6000",
  "input_tokens": 24,
  "output_tokens": 68,
  "total_tokens": 92,
  "model_load_seconds": 7.388,
  "generation_seconds": 2.958,
  "inference_seconds": 2.987,
  "output_tokens_per_second": 22.99,
  "cuda_allocated_gib": 3.083,
  "cuda_peak_allocated_gib": 3.182,
  "cuda_peak_reserved_gib": 3.225,
  "device_total_memory_gib": 47.401,
  "device_free_before_gib": 7.246,
  "device_free_after_gib": 7.121,
  "device_gpu_utilization_mean_percent": 98.4,
  "de